# Supervised Learning: Klassifikation

Klassifikation bedeutet: wir sagen vorher, **zu welcher Gruppe** ein Datenpunkt gehört.

## Inhaltsverzeichnis
1. Der Breast-Cancer-Datensatz
2. KNN — K-Nächste-Nachbarn
3. Overfitting (Überanpassung)
4. Logistische Regression
5. Metriken: Confusion Matrix, Precision, Recall, F1
6. Decision Trees (Entscheidungsbäume)
7. Feature Importance
8. Support Vector Machines (SVM)
9. Cross-Validation (Kreuzvalidierung)


## 1. Der Breast-Cancer-Datensatz

Wir verwenden einen medizinischen Datensatz über Brustkrebs.  
**Aufgabe:** Ist ein Tumor gutartig (benign) oder bösartig (malignant)?  
**Typ:** Binäre Klassifikation (2 Klassen)


In [ ]:
from sklearn.datasets import load_breast_cancer
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Datensatz laden
cancer = load_breast_cancer()

# In DataFrame umwandeln
X = pd.DataFrame(cancer.data, columns=cancer.feature_names)
y = pd.Series(cancer.target, name='label')

print(f"Datensatzgröße: {X.shape}")
print(f"Klassen: {cancer.target_names}")
print(f"Klassenverteilung:")
print(f"  Bösartig (0): {sum(y==0)} Fälle")
print(f"  Gutartig  (1): {sum(y==1)} Fälle")
print()
X.head()

In [ ]:
from sklearn.model_selection import train_test_split

# Immer zuerst aufteilen!
X_train, X_test, y_train, y_test = train_test_split(X, y, random_state=0)
print(f"Training: {X_train.shape[0]} | Test: {X_test.shape[0]}")

## 2. KNN — K-Nächste-Nachbarn

**Idee:** Ein neuer Datenpunkt bekommt die Klasse, die unter seinen k nächsten Nachbarn am häufigsten vorkommt.

**Analogie:** Du ziehst in eine neue Stadt und willst wissen, ob ein Viertel sicher ist. Du fragst die 5 Leute, die am nächsten wohnen. 4 sagen "sicher", 1 sagt "nicht sicher" → Vorhersage: sicher.

**Hyperparameter `n_neighbors` (k):**
- Kleines k (z.B. 1): Sehr flexibel, lernt jeden Datenpunkt auswendig → **Overfitting**
- Großes k (z.B. 30): Sehr glatt, ignoriert Details → **Underfitting**


In [ ]:
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score

# KNN mit k=9
knn = KNeighborsClassifier(n_neighbors=9)
knn.fit(X_train, y_train)

print(f"Trainings-Genauigkeit: {knn.score(X_train, y_train):.2%}")
print(f"Test-Genauigkeit:      {knn.score(X_test, y_test):.2%}")


## 3. Overfitting und Underfitting

**Overfitting (Überanpassung):**
- Modell lernt Trainingsdaten AUSWENDIG, auch das Rauschen
- Traingsgenauigkeit ≈ 100%, Testgenauigkeit viel schlechter
- Wie ein Schüler, der nur die Musterprüfung auswendig lernt, aber echte Prüfungen versagt

**Underfitting (Unteranpassung):**
- Modell ist zu einfach, lernt nicht mal die Trainingsdaten richtig
- Sowohl Trainings- als auch Testgenauigkeit schlecht

**Das Ziel:** Gute Testgenauigkeit = Modell **generalisiert** auf neue Daten!


In [ ]:
# Wie verändert sich k die Genauigkeit?
trainings_genauigkeit = []
test_genauigkeit = []

for k in range(1, 30):
    knn = KNeighborsClassifier(n_neighbors=k)
    knn.fit(X_train, y_train)
    trainings_genauigkeit.append(knn.score(X_train, y_train))
    test_genauigkeit.append(knn.score(X_test, y_test))

plt.figure(figsize=(10, 5))
plt.plot(range(1, 30), trainings_genauigkeit, label="Training", marker='o', markersize=4)
plt.plot(range(1, 30), test_genauigkeit, label="Test", marker='s', markersize=4)
plt.xlabel("k (Anzahl Nachbarn)")
plt.ylabel("Genauigkeit")
plt.title("KNN: Overfitting bei kleinem k")
plt.legend()
plt.grid(True)
plt.show()

bestes_k = test_genauigkeit.index(max(test_genauigkeit)) + 1
print(f"Bestes k: {bestes_k} mit Test-Genauigkeit: {max(test_genauigkeit):.2%}")

## 4. Logistische Regression

**Achtung:** Trotz des Namens ist das ein **Klassifikationsalgorithmus**, kein Regressionsalgorithmus!

**Idee:** Berechnet die **Wahrscheinlichkeit**, dass ein Datenpunkt zu einer Klasse gehört, durch eine S-Kurve (Sigmoid-Funktion). Wenn die Wahrscheinlichkeit > 50% → Klasse 1, sonst Klasse 0.

**Wann gut?**
- Binäre Klassifikation
- Wenn du Wahrscheinlichkeiten brauchst (nicht nur "Ja/Nein")
- Als einfache Baseline


In [ ]:
from sklearn.linear_model import LogisticRegression

log_reg = LogisticRegression(max_iter=10000)  # max_iter erhöhen für Konvergenz
log_reg.fit(X_train, y_train)

print(f"Trainings-Genauigkeit: {log_reg.score(X_train, y_train):.2%}")
print(f"Test-Genauigkeit:      {log_reg.score(X_test, y_test):.2%}")

# Wahrscheinlichkeiten ausgeben (nicht nur "0 oder 1")
wahrscheinlichkeiten = log_reg.predict_proba(X_test[:5])
print()
print("Wahrscheinlichkeiten für die ersten 5 Testfälle:")
for i, (p_boes, p_gut) in enumerate(wahrscheinlichkeiten):
    print(f"  Fall {i+1}: {p_boes:.1%} bösartig, {p_gut:.1%} gutartig")

## 5. Metriken: Mehr als nur Accuracy

**Problem mit Accuracy:** Bei ungleichmäßigen Klassen (z.B. 99% gesund, 1% krank) erreicht man 99% Accuracy, wenn man IMMER "gesund" sagt — aber alle Kranken werden übersehen!

### Confusion Matrix

```
                 Vorhergesagt:
                  Nein    Ja
Tatsächlich: Nein   TN    FP    (FP = False Positive = falscher Alarm)
             Ja     FN    TP    (FN = False Negative = übersehen!)
```

- **TP** (True Positive): Richtig als positiv erkannt
- **TN** (True Negative): Richtig als negativ erkannt  
- **FP** (False Positive): Fälschlicherweise als positiv erkannt → "falscher Alarm"
- **FN** (False Negative): Fälschlicherweise als negativ erkannt → "übersehen"

### Precision vs. Recall

**Precision** = TP / (TP + FP) → "Wie zuverlässig sind positive Vorhersagen?"  
→ Wichtig wenn: Falsche Alarme teuer sind (z.B. Spam-Filter: gute Mails löschen)

**Recall** = TP / (TP + FN) → "Wie viele echte Positive wurden gefunden?"  
→ Wichtig wenn: Übersehene Fälle gefährlich sind (z.B. Krebs-Diagnose!)

**F1-Score** = harmonisches Mittel aus Precision und Recall  
→ Wenn beides gleich wichtig ist


In [ ]:
from sklearn.metrics import confusion_matrix, classification_report
import seaborn as sns

vorhersagen = log_reg.predict(X_test)

# Confusion Matrix visualisieren
cm = confusion_matrix(y_test, vorhersagen)
plt.figure(figsize=(6, 4))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['Bösartig', 'Gutartig'],
            yticklabels=['Bösartig', 'Gutartig'])
plt.xlabel('Vorhergesagt')
plt.ylabel('Tatsächlich')
plt.title('Confusion Matrix - Logistische Regression')
plt.show()

print(classification_report(y_test, vorhersagen, target_names=['Bösartig', 'Gutartig']))

## 6. Decision Trees (Entscheidungsbäume)

**Idee:** Der Algorithmus stellt eine Reihe von Ja/Nein-Fragen und baut daraus einen Baum auf.

**Analogie:** "20 Fragen spielen" — Durch geschicktes Fragen kommt man zur richtigen Antwort.

**Beispiel-Baum:**
```
Ist worst_radius > 16.8?
  ├─ Ja → Bösartig (95% sicher)
  └─ Nein → Ist mean_concave_points > 0.05?
              ├─ Ja → Bösartig
              └─ Nein → Gutartig
```

**Wichtiger Hyperparameter:** `max_depth`
- Kein Limit: Baum wächst bis er jeden Datenpunkt auswendig kennt → starkes Overfitting
- Kleines Limit: Baum ist einfach und generalisiert besser


In [ ]:
from sklearn.tree import DecisionTreeClassifier

# Ohne Limit: Overfitting!
baum_tief = DecisionTreeClassifier(random_state=0)
baum_tief.fit(X_train, y_train)
print("Ohne Tiefenbeschränkung:")
print(f"  Training: {baum_tief.score(X_train, y_train):.2%}")
print(f"  Test:     {baum_tief.score(X_test, y_test):.2%}")

# Mit Limit: Besser generalisiert
baum_flach = DecisionTreeClassifier(max_depth=4, random_state=0)
baum_flach.fit(X_train, y_train)
print()
print("Mit max_depth=4:")
print(f"  Training: {baum_flach.score(X_train, y_train):.2%}")
print(f"  Test:     {baum_flach.score(X_test, y_test):.2%}")

In [ ]:
# max_depth optimieren
tiefen = range(1, 20)
train_scores = []
test_scores = []

for tiefe in tiefen:
    b = DecisionTreeClassifier(max_depth=tiefe, random_state=0)
    b.fit(X_train, y_train)
    train_scores.append(b.score(X_train, y_train))
    test_scores.append(b.score(X_test, y_test))

plt.figure(figsize=(10, 5))
plt.plot(tiefen, train_scores, label="Training", marker='o', markersize=4)
plt.plot(tiefen, test_scores, label="Test", marker='s', markersize=4)
plt.xlabel("max_depth")
plt.ylabel("Genauigkeit")
plt.title("Decision Tree: Overfitting als Funktion der Tiefe")
plt.legend()
plt.grid(True)
plt.show()

## 7. Feature Importance

Ein großer Vorteil von Decision Trees:  
Man kann sehen, welche Merkmale am wichtigsten für die Vorhersage sind!

Das hilft bei der **Interpretation**: "Was treibt eigentlich die Vorhersage an?"


In [ ]:
# Feature Importance des optimierten Baums
bester_baum = DecisionTreeClassifier(max_depth=4, random_state=0)
bester_baum.fit(X_train, y_train)

# Importances ausgeben und sortieren
wichtigkeit = pd.Series(bester_baum.feature_importances_, index=cancer.feature_names)
wichtigkeit = wichtigkeit.sort_values(ascending=True)

plt.figure(figsize=(8, 10))
wichtigkeit.plot(kind='barh', color='steelblue')
plt.xlabel("Wichtigkeit (Feature Importance)")
plt.title("Welche Merkmale sind am wichtigsten?")
plt.tight_layout()
plt.show()

print("Top 5 wichtigste Merkmale:")
print(wichtigkeit.tail(5).sort_values(ascending=False))

## 8. Support Vector Machines (SVM)

**Idee:** Finde die Grenze (Hyperebene), die die Klassen mit dem **größten Abstand** trennt.

**Analogie:** Du zeichnest eine Linie zwischen zwei Gruppen von Punkten — SVM findet die Linie, die so weit wie möglich von beiden Gruppen entfernt ist.

**Besonderheit:** Nur die Punkte nahe der Grenze (die "Support Vectors") beeinflussen die Grenze — alle anderen Punkte sind irrelevant.

**Wann gut?**
- Hochdimensionale Daten (viele Merkmale)
- Klare Trennung zwischen Klassen
- Wenn du nicht zu viele Datenpunkte hast


In [ ]:
from sklearn.svm import LinearSVC

svm = LinearSVC(max_iter=10000, random_state=0)
svm.fit(X_train, y_train)

print(f"Trainings-Genauigkeit: {svm.score(X_train, y_train):.2%}")
print(f"Test-Genauigkeit:      {svm.score(X_test, y_test):.2%}")

## 9. Cross-Validation (Kreuzvalidierung)

**Problem mit einfachem Train/Test-Split:**  
Der zufällige Split kann das Ergebnis beeinflussen. Mit einem anderen `random_state` bekommen wir andere Zahlen.

**Lösung: K-Fold Cross-Validation**
1. Daten in k Teile (Folds) aufteilen (z.B. k=5)
2. k Mal trainieren: immer ein anderer Fold als Test, der Rest als Training
3. Durchschnitt der k Ergebnisse nehmen

**Vorteil:** Zuverlässigere Einschätzung der Modellgüte, nutzt alle Daten effizient.


In [ ]:
from sklearn.model_selection import cross_val_score

# 5-Fold Cross-Validation für verschiedene Algorithmen
modelle = {
    'KNN (k=9)': KNeighborsClassifier(n_neighbors=9),
    'Logistische Regression': LogisticRegression(max_iter=10000),
    'Decision Tree (depth=4)': DecisionTreeClassifier(max_depth=4, random_state=0),
    'SVM': LinearSVC(max_iter=10000, random_state=0)
}

print("5-Fold Cross-Validation Ergebnisse:")
print("-" * 45)
for name, modell in modelle.items():
    scores = cross_val_score(modell, X, y, cv=5)
    print(f"{name:<35} {scores.mean():.2%} ± {scores.std():.2%}")

## Zusammenfassung: Welchen Algorithmus wählen?

| Algorithmus | Stärken | Schwächen |
|------------|---------|-----------|
| **KNN** | Einfach, keine Annahmen | Langsam bei großen Daten, braucht Normalisierung |
| **Logistische Regression** | Schnell, interpretierbar, gibt Wahrscheinlichkeiten | Nur lineare Grenzen |
| **Decision Tree** | Interpretierbar, kein Preprocessing nötig | Overfitting, instabil |
| **SVM** | Gut bei vielen Merkmalen | Langsam bei großen Daten, schwer zu interpretieren |

**Faustregel:** Starte immer mit der **Logistischen Regression** als Baseline. Dann kompliziertere Modelle.
